<a href="https://colab.research.google.com/github/jintubhuyan-2000/Spatial-Gradient-of-Highway-Induced-Land-Cover-Change/blob/main/SECTION_4_11_%E2%80%93_DEVELOPMENT_PRESSURE_AND_TRANSFORMATION_HOTSPOTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# TEZPUR – NORTH LAKHIMPUR HIGHWAY CORRIDOR
# SECTION 4.11 – DEVELOPMENT PRESSURE AND TRANSFORMATION HOTSPOTS
# =============================================================================
#
# PURPOSE
# -------
# Extract complete statistical information from the exported GeoTIFF files for:
#
# Figure 18:
#   TZPR_NLP_Development_Pressure_Score
#
# Figure 19:
#   TZPR_NLP_High_Development_Hotspots
#
# Figure 20:
#   TZPR_NLP_New_BuiltUp_Component
#   TZPR_NLP_Cropland_to_Built_Component
#   TZPR_NLP_Trees_to_Built_Component
#   TZPR_NLP_NDVI_Decline
#
# OUTPUTS
# -------
# 1. Pixel statistics
# 2. Area statistics
# 3. Score distribution
# 4. High-pressure hotspot statistics
# 5. Individual component statistics
# 6. Component overlap
# 7. Number of coincident indicators
# 8. Hotspot intensity
# 9. Percent of corridor affected
# 10. Publication-quality figures
# 11. Derived GeoTIFF products
#
# DEVELOPMENT SCORE FROM GEE
# --------------------------
# New Built-up             = 1
# Cropland → Built         = 2
# Trees → Built            = 2
# NDVI decline             = 1
# NDBI increase            = 1
#
# Maximum theoretical score = 7
# High-development hotspot = score >= 4
#
# =============================================================================


# =============================================================================
# 1. IMPORT LIBRARIES
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from rasterio.warp import reproject, Resampling
from rasterio.transform import xy

warnings.filterwarnings("ignore")


# =============================================================================
# 2. USER SETTINGS
# =============================================================================

# -------------------------------------------------------------------------
# CHANGE THIS TO YOUR GEO-TIFF FOLDER
# -------------------------------------------------------------------------

INPUT_DIR = r"C:\TZPR_NLP_Research\GeoTIFF"

# -------------------------------------------------------------------------
# OUTPUT DIRECTORY
# -------------------------------------------------------------------------

OUTPUT_DIR = os.path.join(
    INPUT_DIR,
    "Development_Pressure_Results"
)

CSV_DIR = os.path.join(
    OUTPUT_DIR,
    "CSV"
)

FIG_DIR = os.path.join(
    OUTPUT_DIR,
    "Figures"
)

TIF_DIR = os.path.join(
    OUTPUT_DIR,
    "GeoTIFF"
)

os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TIF_DIR, exist_ok=True)


# =============================================================================
# 3. INPUT FILES
# =============================================================================

FILES = {

    "development_score":
        "TZPR_NLP_Development_Pressure_Score.tif",

    "high_hotspot":
        "TZPR_NLP_High_Development_Hotspots.tif",

    "new_built":
        "TZPR_NLP_New_BuiltUp_Component.tif",

    "crop_to_built":
        "TZPR_NLP_Cropland_to_Built_Component.tif",

    "tree_to_built":
        "TZPR_NLP_Trees_to_Built_Component.tif",

    "ndvi_decline":
        "TZPR_NLP_NDVI_Decline.tif"
}


# =============================================================================
# 4. FULL PATHS
# =============================================================================

PATHS = {}

for key, filename in FILES.items():

    path = os.path.join(
        INPUT_DIR,
        filename
    )

    PATHS[key] = path


# =============================================================================
# 5. CHECK FILES
# =============================================================================

print("\n" + "=" * 80)
print("CHECKING INPUT FILES")
print("=" * 80)

missing_files = []

for key, path in PATHS.items():

    if os.path.exists(path):

        print(
            f"[FOUND] {os.path.basename(path)}"
        )

    else:

        print(
            f"[MISSING] {os.path.basename(path)}"
        )

        missing_files.append(path)


if missing_files:

    print("\nERROR: The following files are missing:")

    for f in missing_files:
        print("   ", f)

    raise FileNotFoundError(
        "\nPlease place all required GeoTIFF files "
        "inside INPUT_DIR."
    )


# =============================================================================
# 6. READ RASTER
# =============================================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)

        profile = src.profile.copy()

        transform = src.transform

        crs = src.crs

        nodata = src.nodata

        width = src.width

        height = src.height

        resolution = src.res

        bounds = src.bounds

    return {
        "data": data,
        "profile": profile,
        "transform": transform,
        "crs": crs,
        "nodata": nodata,
        "width": width,
        "height": height,
        "resolution": resolution,
        "bounds": bounds
    }


# =============================================================================
# 7. LOAD ALL RASTERS
# =============================================================================

print("\n" + "=" * 80)
print("LOADING GEOTIFF DATA")
print("=" * 80)

rasters = {}

for key, path in PATHS.items():

    print(
        f"Reading: {os.path.basename(path)}"
    )

    rasters[key] = read_raster(path)


# =============================================================================
# 8. DISPLAY RASTER INFORMATION
# =============================================================================

print("\n" + "=" * 80)
print("RASTER INFORMATION")
print("=" * 80)

for key, r in rasters.items():

    print("\n", key)

    print(
        "  Size:",
        r["width"],
        "x",
        r["height"]
    )

    print(
        "  CRS:",
        r["crs"]
    )

    print(
        "  Resolution:",
        r["resolution"]
    )

    print(
        "  Bounds:",
        r["bounds"]
    )

    print(
        "  NoData:",
        r["nodata"]
    )


# =============================================================================
# 9. REFERENCE GRID
# =============================================================================

reference = rasters["development_score"]

ref_data = reference["data"]

ref_transform = reference["transform"]

ref_crs = reference["crs"]

ref_shape = ref_data.shape


# =============================================================================
# 10. CONVERT NODATA TO NAN
# =============================================================================

def clean_raster(raster):

    data = raster["data"].astype(
        np.float32
    )

    nodata = raster["nodata"]

    if nodata is not None:

        data[data == nodata] = np.nan

    return data


# =============================================================================
# 11. ALIGN RASTERS
# =============================================================================

def align_to_reference(
    raster,
    reference_raster
):

    source = clean_raster(raster)

    destination = np.full(
        reference_raster["data"].shape,
        np.nan,
        dtype=np.float32
    )

    reproject(

        source=source,

        destination=destination,

        src_transform=raster["transform"],

        src_crs=raster["crs"],

        dst_transform=reference_raster["transform"],

        dst_crs=reference_raster["crs"],

        resampling=Resampling.nearest,

        src_nodata=np.nan,

        dst_nodata=np.nan
    )

    return destination


# =============================================================================
# 12. PREPARE ALL DATA ON SAME GRID
# =============================================================================

print("\n" + "=" * 80)
print("ALIGNING ALL RASTERS")
print("=" * 80)

data = {}

for key, raster in rasters.items():

    data[key] = align_to_reference(
        raster,
        reference
    )

    print(
        f"[OK] {key}"
    )


# =============================================================================
# 13. BASIC VALID MASK
# =============================================================================

valid_mask = np.isfinite(
    data["development_score"]
)


# =============================================================================
# 14. AREA PER PIXEL
# =============================================================================
#
# For the GeoTIFF exported at 10 m:
#
# 10 m x 10 m = 100 m²
# 1 hectare = 10,000 m²
#
# Therefore:
#
# 1 pixel = 0.01 ha
#
# However, this script calculates area from the raster transform.
#
# =============================================================================

pixel_width = abs(
    ref_transform.a
)

pixel_height = abs(
    ref_transform.e
)

pixel_area_m2 = (
    pixel_width *
    pixel_height
)

pixel_area_ha = (
    pixel_area_m2 /
    10000.0
)

pixel_area_km2 = (
    pixel_area_m2 /
    1e6
)

print("\nPixel area:")
print(
    f"  {pixel_area_m2:.4f} m²"
)

print(
    f"  {pixel_area_ha:.6f} ha"
)

print(
    f"  {pixel_area_km2:.8f} km²"
)


# =============================================================================
# 15. TOTAL STUDY AREA
# =============================================================================

total_valid_pixels = np.sum(
    valid_mask
)

total_area_ha = (
    total_valid_pixels *
    pixel_area_ha
)

total_area_km2 = (
    total_valid_pixels *
    pixel_area_km2
)

print("\nStudy area represented by valid pixels:")

print(
    f"  Pixels: {total_valid_pixels:,}"
)

print(
    f"  Area: {total_area_ha:,.2f} ha"
)

print(
    f"  Area: {total_area_km2:,.4f} km²"
)


# =============================================================================
# 16. DEVELOPMENT SCORE STATISTICS
# =============================================================================

score = data["development_score"]

score_valid = score[
    np.isfinite(score)
]

score_stats = {

    "Metric":
        "Development Pressure Score",

    "Valid_Pixels":
        len(score_valid),

    "Study_Area_ha":
        total_area_ha,

    "Study_Area_km2":
        total_area_km2,

    "Minimum":
        np.min(score_valid),

    "Maximum":
        np.max(score_valid),

    "Mean":
        np.mean(score_valid),

    "Median":
        np.median(score_valid),

    "Std_Dev":
        np.std(score_valid),

    "25th_Percentile":
        np.percentile(score_valid, 25),

    "75th_Percentile":
        np.percentile(score_valid, 75),

    "90th_Percentile":
        np.percentile(score_valid, 90),

    "95th_Percentile":
        np.percentile(score_valid, 95)
}

score_stats_df = pd.DataFrame(
    [score_stats]
)

score_stats_df.to_csv(
    os.path.join(
        CSV_DIR,
        "Development_Pressure_Statistics.csv"
    ),
    index=False
)


# =============================================================================
# 17. DEVELOPMENT SCORE DISTRIBUTION
# =============================================================================

score_rows = []

for s in range(0, 8):

    mask = (
        score == s
    )

    pixels = np.sum(
        mask
    )

    area_ha = (
        pixels *
        pixel_area_ha
    )

    area_km2 = (
        pixels *
        pixel_area_km2
    )

    percentage = (
        area_ha /
        total_area_ha *
        100
    )

    score_rows.append({

        "Development_Score":
            s,

        "Pixel_Count":
            pixels,

        "Area_ha":
            area_ha,

        "Area_km2":
            area_km2,

        "Percent_of_Corridor":
            percentage
    })


score_distribution = pd.DataFrame(
    score_rows
)

score_distribution.to_csv(
    os.path.join(
        CSV_DIR,
        "Development_Pressure_Score_Distribution.csv"
    ),
    index=False
)


# =============================================================================
# 18. SCORE CATEGORY
# =============================================================================

def score_category(value):

    if value == 0:
        return "No pressure"

    elif value == 1:
        return "Very low"

    elif value == 2:
        return "Low"

    elif value == 3:
        return "Moderate"

    elif value == 4:
        return "High"

    elif value == 5:
        return "Very high"

    elif value == 6:
        return "Very high"

    elif value == 7:
        return "Extreme"

    return "Unknown"


score_distribution[
    "Pressure_Category"
] = score_distribution[
    "Development_Score"
].apply(score_category)


score_distribution.to_csv(
    os.path.join(
        CSV_DIR,
        "Development_Pressure_Score_Area.csv"
    ),
    index=False
)


# =============================================================================
# 19. HIGH DEVELOPMENT HOTSPOT
# =============================================================================
#
# According to the GEE script:
#
# developmentScore >= 4
#
# =============================================================================

high_hotspot = (
    score >= 4
)

high_hotspot &= valid_mask

hotspot_pixels = np.sum(
    high_hotspot
)

hotspot_area_ha = (
    hotspot_pixels *
    pixel_area_ha
)

hotspot_area_km2 = (
    hotspot_pixels *
    pixel_area_km2
)

hotspot_percentage = (
    hotspot_area_ha /
    total_area_ha *
    100
)


# =============================================================================
# 20. HOTSPOT STATISTICS
# =============================================================================

hotspot_scores = score[
    high_hotspot
]

hotspot_stats = {

    "Hotspot_Threshold":
        ">=4",

    "Hotspot_Pixels":
        hotspot_pixels,

    "Hotspot_Area_ha":
        hotspot_area_ha,

    "Hotspot_Area_km2":
        hotspot_area_km2,

    "Percent_of_Corridor":
        hotspot_percentage,

    "Mean_Hotspot_Score":
        np.mean(hotspot_scores)
        if len(hotspot_scores) > 0
        else np.nan,

    "Median_Hotspot_Score":
        np.median(hotspot_scores)
        if len(hotspot_scores) > 0
        else np.nan,

    "Maximum_Hotspot_Score":
        np.max(hotspot_scores)
        if len(hotspot_scores) > 0
        else np.nan
}


hotspot_stats_df = pd.DataFrame(
    [hotspot_stats]
)

hotspot_stats_df.to_csv(
    os.path.join(
        CSV_DIR,
        "Development_Hotspot_Statistics.csv"
    ),
    index=False
)


# =============================================================================
# 21. INDIVIDUAL DEVELOPMENT COMPONENTS
# =============================================================================

components = {

    "New_BuiltUp":
        data["new_built"],

    "Cropland_to_Built":
        data["crop_to_built"],

    "Trees_to_Built":
        data["tree_to_built"],

    "NDVI_Decline":
        data["ndvi_decline"]
}


component_rows = []


for name, raster in components.items():

    valid = np.isfinite(
        raster
    )

    # Component rasters are binary
    positive = (
        valid &
        (raster > 0)
    )

    pixels = np.sum(
        positive
    )

    area_ha = (
        pixels *
        pixel_area_ha
    )

    area_km2 = (
        pixels *
        pixel_area_km2
    )

    percent = (
        area_ha /
        total_area_ha *
        100
    )

    # Mean component value
    values = raster[
        valid
    ]

    component_rows.append({

        "Component":
            name,

        "Valid_Pixels":
            np.sum(valid),

        "Positive_Pixels":
            pixels,

        "Area_ha":
            area_ha,

        "Area_km2":
            area_km2,

        "Percent_of_Corridor":
            percent,

        "Mean_Value":
            np.nanmean(values),

        "Median_Value":
            np.nanmedian(values),

        "Minimum":
            np.nanmin(values),

        "Maximum":
            np.nanmax(values)
    })


component_stats = pd.DataFrame(
    component_rows
)

component_stats.to_csv(
    os.path.join(
        CSV_DIR,
        "Development_Component_Statistics.csv"
    ),
    index=False
)


# =============================================================================
# 22. COMPONENT BINARY MASKS
# =============================================================================

new_built_mask = (
    data["new_built"] > 0
)

crop_built_mask = (
    data["crop_to_built"] > 0
)

tree_built_mask = (
    data["tree_to_built"] > 0
)

ndvi_decline_mask = (
    data["ndvi_decline"] > 0
)


# =============================================================================
# 23. COMPONENT OVERLAP COUNT
# =============================================================================
#
# 0 = none
# 1 = one indicator
# 2 = two indicators
# 3 = three indicators
# 4 = all four indicators
#
# =============================================================================

component_count = (

    new_built_mask.astype(np.uint8)

    +

    crop_built_mask.astype(np.uint8)

    +

    tree_built_mask.astype(np.uint8)

    +

    ndvi_decline_mask.astype(np.uint8)
)


component_count = np.where(
    valid_mask,
    component_count,
    255
).astype(np.uint8)


# =============================================================================
# 24. SAVE COMPONENT COUNT GEOTIFF
# =============================================================================

count_profile = reference["profile"].copy()

count_profile.update(

    dtype="uint8",

    count=1,

    nodata=255,

    compress="lzw"
)


count_tif = os.path.join(
    TIF_DIR,
    "Development_Component_Count.tif"
)


with rasterio.open(
    count_tif,
    "w",
    **count_profile
) as dst:

    dst.write(
        component_count,
        1
    )


# =============================================================================
# 25. OVERLAP STATISTICS
# =============================================================================

overlap_rows = []


for n in range(0, 5):

    mask = (
        component_count == n
    )

    pixels = np.sum(
        mask
    )

    area_ha = (
        pixels *
        pixel_area_ha
    )

    area_km2 = (
        pixels *
        pixel_area_km2
    )

    percent = (
        area_ha /
        total_area_ha *
        100
    )

    overlap_rows.append({

        "Number_of_Coincident_Indicators":
            n,

        "Pixel_Count":
            pixels,

        "Area_ha":
            area_ha,

        "Area_km2":
            area_km2,

        "Percent_of_Corridor":
            percent
    })


overlap_stats = pd.DataFrame(
    overlap_rows
)

overlap_stats.to_csv(
    os.path.join(
        CSV_DIR,
        "Component_Overlap_Statistics.csv"
    ),
    index=False
)


# =============================================================================
# 26. HIGH HOTSPOT OVERLAP
# =============================================================================

hotspot_component_count = np.where(

    high_hotspot,

    component_count,

    255

).astype(np.uint8)


hotspot_overlap_rows = []


for n in range(0, 5):

    mask = (
        hotspot_component_count == n
    )

    pixels = np.sum(
        mask
    )

    area_ha = (
        pixels *
        pixel_area_ha
    )

    area_km2 = (
        pixels *
        pixel_area_km2
    )

    if hotspot_pixels > 0:

        hotspot_percent = (
            area_ha /
            hotspot_area_ha *
            100
        )

    else:

        hotspot_percent = 0


    hotspot_overlap_rows.append({

        "Indicators_Coinciding":
            n,

        "Hotspot_Pixels":
            pixels,

        "Area_ha":
            area_ha,

        "Area_km2":
            area_km2,

        "Percent_of_Hotspots":
            hotspot_percent
    })


hotspot_overlap_stats = pd.DataFrame(
    hotspot_overlap_rows
)

hotspot_overlap_stats.to_csv(
    os.path.join(
        CSV_DIR,
        "Hotspot_Overlap_Statistics.csv"
    ),
    index=False
)


# =============================================================================
# 27. HIGH-DEVELOPMENT HOTSPOT CLASSIFICATION
# =============================================================================
#
# 0 = no hotspot
# 1 = score 4
# 2 = score 5
# 3 = score 6
# 4 = score 7
#
# =============================================================================

hotspot_class = np.where(

    high_hotspot,

    score,

    0
)


hotspot_class = np.where(

    valid_mask,

    hotspot_class,

    255
)


hotspot_class = hotspot_class.astype(
    np.uint8
)


# =============================================================================
# 28. SAVE HOTSPOT GEOTIFF
# =============================================================================

hotspot_profile = reference["profile"].copy()

hotspot_profile.update(

    dtype="uint8",

    count=1,

    nodata=255,

    compress="lzw"
)


hotspot_tif = os.path.join(

    TIF_DIR,

    "Development_Pressure_Hotspot.tif"
)


with rasterio.open(

    hotspot_tif,

    "w",

    **hotspot_profile

) as dst:

    dst.write(
        hotspot_class,
        1
    )


# =============================================================================
# 29. HOTSPOT INTENSITY STATISTICS
# =============================================================================

intensity_rows = []


for s in range(4, 8):

    mask = (
        score == s
    )

    pixels = np.sum(
        mask
    )

    area_ha = (
        pixels *
        pixel_area_ha
    )

    area_km2 = (
        pixels *
        pixel_area_km2
    )

    percent_corridor = (
        area_ha /
        total_area_ha *
        100
    )

    percent_hotspot = (

        area_ha /
        hotspot_area_ha *
        100

        if hotspot_area_ha > 0

        else 0
    )


    intensity_rows.append({

        "Development_Score":
            s,

        "Pixels":
            pixels,

        "Area_ha":
            area_ha,

        "Area_km2":
            area_km2,

        "Percent_of_Corridor":
            percent_corridor,

        "Percent_of_Hotspots":
            percent_hotspot
    })


hotspot_intensity = pd.DataFrame(
    intensity_rows
)

hotspot_intensity.to_csv(
    os.path.join(
        CSV_DIR,
        "Hotspot_Intensity_Statistics.csv"
    ),
    index=False
)


# =============================================================================
# 30. SUMMARY TABLE
# =============================================================================

summary = {

    "Total_Corridor_Area_ha":
        total_area_ha,

    "Total_Corridor_Area_km2":
        total_area_km2,

    "Mean_Development_Score":
        np.mean(score_valid),

    "Median_Development_Score":
        np.median(score_valid),

    "Maximum_Development_Score":
        np.max(score_valid),

    "High_Hotspot_Area_ha":
        hotspot_area_ha,

    "High_Hotspot_Area_km2":
        hotspot_area_km2,

    "High_Hotspot_Percent":
        hotspot_percentage,

    "New_Built_Area_ha":
        component_stats.loc[
            component_stats["Component"]
            == "New_BuiltUp",
            "Area_ha"
        ].iloc[0],

    "Cropland_to_Built_Area_ha":
        component_stats.loc[
            component_stats["Component"]
            == "Cropland_to_Built",
            "Area_ha"
        ].iloc[0],

    "Trees_to_Built_Area_ha":
        component_stats.loc[
            component_stats["Component"]
            == "Trees_to_Built",
            "Area_ha"
        ].iloc[0],

    "NDVI_Decline_Area_ha":
        component_stats.loc[
            component_stats["Component"]
            == "NDVI_Decline",
            "Area_ha"
        ].iloc[0],

    "Two_or_More_Indicators_Area_ha":
        overlap_stats.loc[
            overlap_stats[
                "Number_of_Coincident_Indicators"
            ] >= 2,
            "Area_ha"
        ].sum(),

    "Three_or_More_Indicators_Area_ha":
        overlap_stats.loc[
            overlap_stats[
                "Number_of_Coincident_Indicators"
            ] >= 3,
            "Area_ha"
        ].sum(),

    "All_Four_Indicators_Area_ha":
        overlap_stats.loc[
            overlap_stats[
                "Number_of_Coincident_Indicators"
            ] == 4,
            "Area_ha"
        ].sum()
}


summary_df = pd.DataFrame(
    [summary]
)

summary_df.to_csv(
    os.path.join(
        CSV_DIR,
        "Development_Pressure_Summary.csv"
    ),
    index=False
)


# =============================================================================
# 31. FIGURE 18 – DEVELOPMENT PRESSURE SCORE
# =============================================================================

plt.figure(
    figsize=(12, 9)
)

display_score = np.where(
    valid_mask,
    score,
    np.nan
)

plt.imshow(
    display_score,
    cmap="YlOrRd",
    vmin=0,
    vmax=7
)

cbar = plt.colorbar(
    fraction=0.035,
    pad=0.04
)

cbar.set_label(
    "Integrated Development Pressure Score",
    fontsize=11
)

plt.title(
    "Figure 18. Spatial Distribution of Integrated Development Pressure\n"
    "within the Tezpur–North Lakhimpur Highway Corridor",
    fontsize=13,
    fontweight="bold"
)

plt.axis("off")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Figure_18_Development_Pressure.png"
    ),
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 32. FIGURE 19 – HIGH DEVELOPMENT HOTSPOTS
# =============================================================================

plt.figure(
    figsize=(12, 9)
)

hotspot_display = np.where(
    high_hotspot,
    1,
    np.nan
)

plt.imshow(
    hotspot_display,
    cmap="Reds",
    vmin=0,
    vmax=1
)

plt.title(
    "Figure 19. High-Development-Pressure Hotspots\n"
    "Identified from Integrated Landscape Transformation Indicators",
    fontsize=13,
    fontweight="bold"
)

plt.axis("off")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Figure_19_High_Development_Hotspots.png"
    ),
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 33. FIGURE 20 – INDIVIDUAL DEVELOPMENT COMPONENTS
# =============================================================================

component_plot_data = [

    (
        "New Built-up",
        new_built_mask
    ),

    (
        "Cropland → Built",
        crop_built_mask
    ),

    (
        "Trees → Built",
        tree_built_mask
    ),

    (
        "NDVI Decline",
        ndvi_decline_mask
    )
]


for name, mask in component_plot_data:

    plt.figure(
        figsize=(10, 8)
    )

    display = np.where(
        mask,
        1,
        np.nan
    )

    plt.imshow(
        display,
        cmap="Reds",
        vmin=0,
        vmax=1
    )

    plt.title(
        name,
        fontsize=13,
        fontweight="bold"
    )

    plt.axis("off")

    plt.tight_layout()

    safe_name = (
        name
        .replace("→", "to")
        .replace(" ", "_")
    )

    plt.savefig(
        os.path.join(
            FIG_DIR,
            f"Figure_20_{safe_name}.png"
        ),
        dpi=400,
        bbox_inches="tight"
    )

    plt.close()


# =============================================================================
# 34. COMPONENT AREA COMPARISON
# =============================================================================

plt.figure(
    figsize=(10, 6)
)

plt.bar(

    component_stats["Component"],

    component_stats["Area_ha"]
)

plt.ylabel(
    "Area (ha)"
)

plt.xlabel(
    "Development-pressure component"
)

plt.title(
    "Area affected by individual development-pressure components",
    fontsize=13,
    fontweight="bold"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Component_Area_Comparison.png"
    ),
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 35. COMPONENT OVERLAP FIGURE
# =============================================================================

overlap_plot = overlap_stats[
    overlap_stats[
        "Number_of_Coincident_Indicators"
    ] > 0
]


plt.figure(
    figsize=(9, 6)
)

plt.bar(

    overlap_plot[
        "Number_of_Coincident_Indicators"
    ].astype(str),

    overlap_plot["Area_ha"]
)

plt.xlabel(
    "Number of coincident transformation indicators"
)

plt.ylabel(
    "Area (ha)"
)

plt.title(
    "Spatial overlap of development-pressure indicators",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Component_Overlap.png"
    ),
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 36. HOTSPOT SCORE DISTRIBUTION
# =============================================================================

plt.figure(
    figsize=(9, 6)
)

plt.bar(

    hotspot_intensity[
        "Development_Score"
    ].astype(str),

    hotspot_intensity[
        "Area_ha"
    ]
)

plt.xlabel(
    "Development pressure score"
)

plt.ylabel(
    "Hotspot area (ha)"
)

plt.title(
    "Distribution of high-development-pressure hotspot intensity",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Hotspot_Score_Distribution.png"
    ),
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 37. PRINT FINAL RESULTS
# =============================================================================

print("\n")
print("=" * 80)
print("DEVELOPMENT PRESSURE ANALYSIS COMPLETE")
print("=" * 80)

print("\n--- OVERALL DEVELOPMENT PRESSURE ---")

print(
    f"Total corridor area: "
    f"{total_area_ha:,.2f} ha"
)

print(
    f"Mean development score: "
    f"{np.mean(score_valid):.3f}"
)

print(
    f"Median development score: "
    f"{np.median(score_valid):.3f}"
)

print(
    f"Maximum development score: "
    f"{np.max(score_valid):.0f}"
)


print("\n--- HIGH DEVELOPMENT HOTSPOTS ---")

print(
    f"Hotspot pixels: "
    f"{hotspot_pixels:,}"
)

print(
    f"Hotspot area: "
    f"{hotspot_area_ha:,.2f} ha"
)

print(
    f"Hotspot area: "
    f"{hotspot_area_km2:,.4f} km²"
)

print(
    f"Corridor affected: "
    f"{hotspot_percentage:.2f}%"
)


print("\n--- INDIVIDUAL COMPONENTS ---")

for _, row in component_stats.iterrows():

    print(
        f"{row['Component']:25s} "
        f"{row['Area_ha']:,.2f} ha "
        f"({row['Percent_of_Corridor']:.2f}%)"
    )


print("\n--- INDICATOR COINCIDENCE ---")

for _, row in overlap_stats.iterrows():

    if row[
        "Number_of_Coincident_Indicators"
    ] > 0:

        print(

            f"{int(row['Number_of_Coincident_Indicators'])} "
            f"indicator(s): "

            f"{row['Area_ha']:,.2f} ha "
            f"({row['Percent_of_Corridor']:.2f}%)"
        )


print("\n--- CRITICAL TRANSFORMATION AREAS ---")

print(

    "Two or more indicators: "

    f"{summary['Two_or_More_Indicators_Area_ha']:,.2f} ha"
)

print(

    "Three or more indicators: "

    f"{summary['Three_or_More_Indicators_Area_ha']:,.2f} ha"
)

print(

    "All four indicators: "

    f"{summary['All_Four_Indicators_Area_ha']:,.2f} ha"
)


print("\n--- OUTPUT LOCATION ---")

print(
    OUTPUT_DIR
)

print("\nCSV files:")
print(
    CSV_DIR
)

print("\nFigures:")
print(
    FIG_DIR
)

print("\nGeoTIFF files:")
print(
    TIF_DIR
)

print("\n" + "=" * 80)
print("ALL DEVELOPMENT-PRESSURE RESULTS GENERATED")
print("=" * 80)